# Mastering de audio con LGMDM

Cuaderno para cargar un audio, iniciar sesión en la API, ejecutar el mastering síncrono y escuchar/descargar el resultado.

**Requisitos:** la API debe ser accesible desde Colab mediante HTTPS. Si el despliegue publica la API bajo `/api`, incluí ese prefijo en `API_BASE` (por ejemplo, `https://tu-dominio.example/api`). `localhost` de tu computadora no es accesible desde Colab.

Las credenciales y el token se mantienen en la sesión actual del kernel y no se escriben en el cuaderno. El endpoint `/master/sync` ofrece controles generales de mastering; este cuaderno no configura todos los controles DSP avanzados de la interfaz.

In [ ]:
from getpass import getpass
from email.message import Message
from pathlib import Path
from urllib.parse import urlparse
import mimetypes
import requests

API_BASE = input('URL base de la API (HTTPS, incluir /api si corresponde): ').strip().rstrip('/')
parsed_api = urlparse(API_BASE)
if parsed_api.scheme != 'https' or not parsed_api.netloc:
    raise ValueError('Usá una URL HTTPS válida y accesible desde Google Colab.')

REQUEST_TIMEOUT = (30, 1800)  # conexión, lectura; hasta 30 min para masterizar

In [ ]:
def ensure_success(response, operation):
    if response.ok:
        return
    detail = ''
    try:
        payload = response.json()
        detail = payload.get('detail', '') if isinstance(payload, dict) else ''
        if isinstance(detail, list):
            detail = '; '.join(str(item.get('msg', item)) if isinstance(item, dict) else str(item) for item in detail)
    except (ValueError, requests.exceptions.JSONDecodeError):
        pass
    message = f'{operation}: HTTP {response.status_code}'
    if detail:
        message += f' - {detail}'
    raise RuntimeError(message)

email = input('Correo de la cuenta LGMDM: ').strip()
password = getpass('Contraseña (no se muestra): ')
session = requests.Session()
try:
    login_response = session.post(
        f'{API_BASE}/auth/login',
        json={'email': email, 'password': password},
        timeout=(15, 45),
    )
finally:
    del password
ensure_success(login_response, 'No se pudo iniciar sesión')
login_data = login_response.json()
access_token = login_data.get('access_token') if isinstance(login_data, dict) else None
if access_token:
    session.headers['Authorization'] = f'Bearer {access_token}'
if not access_token and not session.cookies:
    raise RuntimeError('El servidor no entregó un token ni una cookie de sesión.')
del access_token, login_data
print('Sesión iniciada. El token y la cookie permanecen en memoria.')

In [ ]:
from google.colab import files

uploaded_files = files.upload()
if not uploaded_files:
    raise RuntimeError('No se seleccionó ningún archivo.')

supported_extensions = {'.wav', '.mp3', '.flac', '.ogg', '.aiff', '.aif'}
audio_candidates = [name for name in uploaded_files if Path(name).suffix.lower() in supported_extensions]
if not audio_candidates:
    raise ValueError('Formato no admitido. Elegí WAV, MP3, FLAC, OGG o AIFF.')
if len(audio_candidates) > 1:
    print(f'Se usará el primer archivo de audio: {audio_candidates[0]}')
input_filename = audio_candidates[0]
input_audio = uploaded_files[input_filename]
if not input_audio:
    raise ValueError('El archivo seleccionado está vacío.')
print(f'Archivo listo: {input_filename} ({len(input_audio) / (1024**2):.1f} MiB)')

## Ajustes del mastering

Editá la celda siguiente antes de ejecutarla. `loudness_target=None` deja el destino manual; las plataformas disponibles son Spotify, YouTube, Apple Music, Tidal, club y CD. Si indicás `platform_target`, el backend aplicará el destino de esa plataforma.

In [ ]:
OUTPUT_FORMAT = 'wav'       # wav, flac o mp3
OUTPUT_BIT_DEPTH = 24      # 16, 24 o 32
PLATFORM_TARGET = None     # None, 'spotify', 'youtube', 'apple_music', 'tidal', 'club' o 'cd'
LOUDNESS_TARGET = -14.0    # LUFS o None
HEADROOM_DB = -1.0         # rango backend: -3.0 a 0.0 dB
CEILING_DB = -0.3          # rango backend: -1.0 a 0.0 dB

allowed_platforms = {None, 'spotify', 'youtube', 'apple_music', 'tidal', 'club', 'cd'}
if OUTPUT_FORMAT not in {'wav', 'flac', 'mp3'}:
    raise ValueError('OUTPUT_FORMAT debe ser wav, flac o mp3.')
if OUTPUT_BIT_DEPTH not in {16, 24, 32}:
    raise ValueError('OUTPUT_BIT_DEPTH debe ser 16, 24 o 32.')
if PLATFORM_TARGET not in allowed_platforms:
    raise ValueError(f'PLATFORM_TARGET debe ser uno de: {sorted(str(x) for x in allowed_platforms)}')
if LOUDNESS_TARGET is not None and not -30.0 <= float(LOUDNESS_TARGET) <= -4.0:
    raise ValueError('LOUDNESS_TARGET debe estar entre -30 y -4 LUFS, o ser None.')
if not -3.0 <= float(HEADROOM_DB) <= 0.0:
    raise ValueError('HEADROOM_DB debe estar entre -3 y 0 dB.')
if not -1.0 <= float(CEILING_DB) <= 0.0:
    raise ValueError('CEILING_DB debe estar entre -1 y 0 dB.')
print('Ajustes válidos.')

In [ ]:
query_params = {
    'output_bit_depth': OUTPUT_BIT_DEPTH,
    'headroom_db': HEADROOM_DB,
    'ceiling_db': CEILING_DB,
}
if PLATFORM_TARGET is not None:
    query_params['platform_target'] = PLATFORM_TARGET
if LOUDNESS_TARGET is not None:
    query_params['loudness_target'] = LOUDNESS_TARGET

mime_type = mimetypes.guess_type(input_filename)[0] or 'application/octet-stream'
print('Procesando en el servidor; el tiempo depende de la duración del audio...')
try:
    mastering_response = session.post(
        f'{API_BASE}/master/sync',
        params=query_params,
        data={'output_format': OUTPUT_FORMAT},
        files={'file': (input_filename, input_audio, mime_type)},
        timeout=REQUEST_TIMEOUT,
    )
except requests.exceptions.Timeout as exc:
    raise RuntimeError('Se agotó el tiempo de espera. El servidor puede seguir ocupado; revisá su estado antes de reenviar el audio.') from exc
,
ensure_success(mastering_response, 'Falló el mastering')
mastered_audio = mastering_response.content
if not mastered_audio:
    raise RuntimeError('El servidor respondió sin contenido de audio.')

content_disposition = Message()
content_disposition['content-disposition'] = mastering_response.headers.get('Content-Disposition', '')
output_filename = content_disposition.get_filename() or f'{Path(input_filename).stem}_mastered.{OUTPUT_FORMAT}'
output_filename = Path(output_filename).name
output_path = Path.cwd() / output_filename
output_path.write_bytes(mastered_audio)
print(f'Máster recibido: {output_path} ({len(mastered_audio) / (1024**2):.1f} MiB)')

In [ ]:
from IPython.display import Audio, display
from google.colab import files

display(Audio(filename=str(output_path)))
files.download(str(output_path))

## Sesión

Al terminar, podés limpiar los objetos sensibles y cerrar la sesión del kernel. El cuaderno no persiste el token ni las credenciales.

In [ ]:
session.close()
for variable_name in ('email', 'session', 'login_response', 'mastering_response', 'input_audio', 'uploaded_files', 'mastered_audio'):
    globals().pop(variable_name, None)
print('Sesión HTTP cerrada.')